# DataFrames de panel PISA y deberes

Se combinan las medias de las asignaturas con las variables de deberes por estrato y por pais.

In [1]:
from pathlib import Path
import pandas as pd
import gc
import sys
import os

#Indicamos el workspace principal (dos niveles atrás)
sys.path.append(os.path.abspath('../../'))


from src.utils_pisa import *


%load_ext autoreload
%autoreload 2


ruta_datos = Path('../../data/processed')

establecer_semilla(42)
pd.set_option('display.float_format', lambda valor: f'{valor:.2f}')


✅ Semillas del proyecto establecidas con éxito (seed=42).


## Panel por pais y estrato

La union usa `CNT`, `STRATUM` y `year` para asociar las observaciones del mismo pais, estrato y edicion.

In [2]:
df_medias_estrato = pd.read_parquet(
    ruta_datos / 'pisa_medias_asignaturas_estrato_anual.parquet'
)
df_deberes_strat = pd.read_parquet(
    ruta_datos / 'deberes_strat.parquet'
)

df_panel_estrato = df_medias_estrato.merge(
    df_deberes_strat,
    on=['CNT', 'STRATUM', 'year'],
    how='inner',
    validate='one_to_one',
)
df_panel_estrato = df_panel_estrato[
    ['CNT', 'STRATUM', 'year']
    + [col for col in df_panel_estrato.columns if col not in ['CNT', 'STRATUM', 'year']]
]

display(df_panel_estrato.head())
print(f'Dimensiones del panel por estrato: {df_panel_estrato.shape}')

,CNT,STRATUM,year,media_math_estrato,media_read_estrato,media_scie_estrato,tiempo_deberes_totales_diarios,dispone_personal_apoyo
0,USA,USA08,2022,476.10,520.35,512.01,3.07,1.00
1,USA,USA07,2022,539.40,535.90,592.47,3.40,NaN
2,USA,USA06,2022,454.64,493.90,490.45,2.55,1.07
3,USA,USA05,2022,476.43,507.31,490.59,3.05,1.00
4,USA,USA04,2022,469.99,503.32,504.79,2.60,1.09


Dimensiones del panel por estrato: (2328, 8)


## Panel por pais

La union usa `CNT` y `year` para asociar las observaciones del mismo pais y edicion.

In [3]:
df_medias_pais = pd.read_parquet(
    ruta_datos / 'pisa_medias_asignaturas_pais_anual.parquet'
)
df_deberes_cnt = pd.read_parquet(
    ruta_datos / 'deberes_cnt.parquet'
)

df_panel_pais = df_medias_pais.merge(
    df_deberes_cnt,
    on=['CNT', 'year'],
    how='inner',
    validate='one_to_one',
)
df_panel_pais = df_panel_pais[
    ['CNT', 'year']
    + [col for col in df_panel_pais.columns if col not in ['CNT', 'year']]
]

display(df_panel_pais.head())
print(f'Dimensiones del panel por pais: {df_panel_pais.shape}')

,CNT,year,media_math_pais,media_read_pais,media_scie_pais,tiempo_deberes_totales_diarios,dispone_personal_apoyo
0,USA,2022,464.89,503.94,499.41,2.73,1.05
1,TUR,2022,453.15,456.08,475.94,3.14,1.62
2,SWE,2022,481.77,486.98,493.55,2.44,1.07
3,SVN,2022,484.53,468.54,499.96,2.44,1.54
4,SVK,2022,463.99,446.86,462.27,2.35,1.30


Dimensiones del panel por pais: (132, 7)


---

# AGREGACION DE LOS **DATOS ECONOMICOS SIMULADOS**

In [4]:
ruta_datos_simulados = Path('../../notebooks/fase_2_feature_engineering')
#pisa_datos_economicos_simulados

In [11]:
def ordenar_columnas_panel(df):
    claves = [col for col in ['CNT', 'STRATUM', 'year'] if col in df.columns]
    medias = [
        col for col in [
            'media_math_pais', 'media_read_pais', 'media_scie_pais',
            'media_math_estrato', 'media_read_estrato', 'media_scie_estrato',
        ]
        if col in df.columns
    ]
    resto = [col for col in df.columns if col not in claves + medias]
    return df[claves + medias + resto]


# Paneles en formato largo
df_panel_estrato = ordenar_columnas_panel(df_panel_estrato)
df_panel_estrato_final = ordenar_columnas_panel(df_panel_estrato_final)
df_panel_pais = ordenar_columnas_panel(df_panel_pais)
df_pais_transaccional = ordenar_columnas_panel(df_pais_transaccional)


# Panel por país en formato ancho: medias primero, agrupadas por asignatura
if 'df_pais_ancho' in globals():
    orden_medias = [
        col
        for prefijo in ['media_math_pais_', 'media_read_pais_', 'media_scie_pais_']
        for col in df_pais_ancho.columns
        if col.startswith(prefijo)
    ]
    claves = ['CNT'] if 'CNT' in df_pais_ancho.columns else []
    resto = [
        col for col in df_pais_ancho.columns
        if col not in claves + orden_medias
    ]
    df_pais_ancho = df_pais_ancho[claves + orden_medias + resto]

In [20]:
# ... [Tu código actual que genera df_panel_estrato y df_panel_pais] ...

# =====================================================================
# INTEGRACIÓN Y PIVOTADO DE VARIABLES ECONÓMICAS ALEATORIAS
# =====================================================================

# 1. Leemos el archivo parquet base aleatorio
df_macro_base = pd.read_parquet(ruta_datos_simulados/ 'pisa_datos_economicos_simulados.parquet')

# Aseguramos tipos uniformes para los cruces
df_macro_base['year'] = df_macro_base['year'].astype(int)
df_panel_estrato['year'] = df_panel_estrato['year'].astype(int)
df_panel_pais['year'] = df_panel_pais['year'].astype(int)


# ---------------------------------------------------------------------
# UNIÓN A: TABLA LARGA POR ESTRATOS (Formato transaccional para modelos)
# ---------------------------------------------------------------------
df_panel_estrato_final = df_panel_estrato.merge(
    df_macro_base,
    on=['CNT', 'year'],
    how='inner'
)

columnas_macro = ['PIB_per_capita_PPP', 'Gasto_Alumno_PPP', 'Gasto_Educacion_PIB', 'Indice_Gini', 'ESCS_media_pais_anual']
df_panel_estrato_final = df_panel_estrato_final[
    ['CNT', 'STRATUM', 'year'] 
    + [col for col in df_panel_estrato_final.columns if col not in [['CNT', 'STRATUM', 'year']]+ columnas_macro]+ columnas_macro
]

print("=== 1. PANEL POR ESTRATO FINAL COMPLETO (TABLA LARGA) ===")
display(df_panel_estrato_final.head())
print(f'Dimensiones del panel largo: {df_panel_estrato_final.shape}\n')


# ---------------------------------------------------------------------
# UNIÓN B: TABLA POR PAÍS (Formato ANCHO como hacías en deberes)
# ---------------------------------------------------------------------
# Primero unimos tus datos de df_panel_pais con la matriz macro base
df_pais_transaccional = df_panel_pais.merge(
    df_macro_base,
    on=['CNT', 'year'],
    how='inner'
)

# Identificamos todas las columnas que queremos pivotar (las tuyas originales + las nuevas económicas)
columnas_a_pivotar = [col for col in df_pais_transaccional.columns if col not in ['CNT', 'year']]

# Aplicamos tu lógica exacta: pivotamos para que los países sean filas y los años columnas
df_pais_ancho = df_pais_transaccional.pivot(
    index='CNT',
    columns='year',
    values=columnas_a_pivotar
)

# Aplanamos el nivel de las columnas para que quede limpio de leer (ej: PIB_per_capita_PPP_2022)
df_pais_ancho.columns = [f'{var}_{anio}' for var, anio in df_pais_ancho.columns]
df_pais_ancho = df_pais_ancho.reset_index()




=== 1. PANEL POR ESTRATO FINAL COMPLETO (TABLA LARGA) ===


,CNT,STRATUM,year,CNT,STRATUM,year,media_math_estrato,media_read_estrato,media_scie_estrato,tiempo_deberes_totales_diarios,dispone_personal_apoyo,PIB_per_capita_PPP,Gasto_Alumno_PPP,Gasto_Educacion_PIB,Indice_Gini,ESCS_media_pais_anual
0,USA,USA08,2022,USA,USA08,2022,476.10,520.35,512.01,3.07,1.00,24568.52,56890.87,4.63,39.15,0.21
1,USA,USA07,2022,USA,USA07,2022,539.40,535.90,592.47,3.40,NaN,24568.52,56890.87,4.63,39.15,0.21
2,USA,USA06,2022,USA,USA06,2022,454.64,493.90,490.45,2.55,1.07,24568.52,56890.87,4.63,39.15,0.21
3,USA,USA05,2022,USA,USA05,2022,476.43,507.31,490.59,3.05,1.00,24568.52,56890.87,4.63,39.15,0.21
4,USA,USA04,2022,USA,USA04,2022,469.99,503.32,504.79,2.60,1.09,24568.52,56890.87,4.63,39.15,0.21


Dimensiones del panel largo: (2328, 16)



In [16]:
print("=== 2. PANEL POR PAÍS FINAL COMPLETO (FORMATO ANCHO PIVOTADO) ===")
display(df_pais_ancho)
print(f'Dimensiones del panel ancho: {df_pais_ancho.shape}')

=== 2. PANEL POR PAÍS FINAL COMPLETO (FORMATO ANCHO PIVOTADO) ===


,CNT,media_math_pais_2012,media_math_pais_2015,media_math_pais_2018,media_math_pais_2022,media_read_pais_2012,media_read_pais_2015,media_read_pais_2018,media_read_pais_2022,media_scie_pais_2012,...,Gasto_Educacion_PIB_2018,Gasto_Educacion_PIB_2022,Indice_Gini_2012,Indice_Gini_2015,Indice_Gini_2018,Indice_Gini_2022,ESCS_media_pais_anual_2012,ESCS_media_pais_anual_2015,ESCS_media_pais_anual_2018,ESCS_media_pais_anual_2022
0,AUS,504.15,493.90,491.36,487.08,511.80,502.90,502.63,498.05,521.49,...,6.75,5.36,37.97,38.03,28.31,33.80,-1.03,0.62,-0.95,-0.63
1,AUT,505.54,496.74,498.94,487.27,489.61,484.87,484.39,480.41,505.78,...,3.29,3.44,32.16,37.81,46.72,40.11,-0.13,-1.36,1.40,-0.18
2,BEL,514.53,506.98,508.07,489.49,508.62,498.52,492.86,478.85,504.87,...,7.23,3.88,45.73,36.67,45.37,24.13,-0.72,-0.95,0.29,-0.52
3,CAN,518.07,515.65,512.02,496.95,523.12,526.67,520.09,507.13,525.44,...,3.02,3.33,31.92,24.86,43.39,31.96,-0.66,1.46,0.62,-1.15
4,CHE,530.93,521.25,515.31,507.99,509.04,492.20,483.93,483.33,515.30,...,6.42,4.92,24.59,45.18,37.03,23.64,-0.57,-0.08,0.81,-1.18
5,CHL,422.63,422.67,417.41,411.70,441.40,458.57,452.27,447.98,444.93,...,7.18,3.84,35.71,28.72,43.20,45.31,1.22,-1.27,0.40,0.12
6,CZE,498.96,492.33,499.47,487.00,492.89,487.25,490.22,488.60,508.30,...,3.54,6.16,25.75,23.17,31.44,32.09,-0.82,0.03,1.33,1.42
7,DEU,513.53,505.97,500.04,474.83,507.68,509.10,498.28,479.79,524.12,...,3.65,6.43,30.52,24.29,35.24,28.94,-0.65,-0.66,1.46,0.68
8,DNK,500.03,511.09,509.40,489.27,496.13,499.81,501.13,488.80,498.47,...,5.30,4.74,36.39,24.02,28.66,46.42,-1.23,0.27,0.44,-1.09
9,ESP,484.32,485.84,481.39,473.14,487.94,495.58,476.54,474.31,496.45,...,7.05,7.04,44.93,36.24,38.83,45.18,-0.73,-0.77,-0.48,0.84


Dimensiones del panel ancho: (33, 41)



---


In [14]:
df_panel_estrato_final.columns

Index(['CNT', 'STRATUM', 'year', 'PIB_per_capita_PPP', 'Gasto_Alumno_PPP',
       'Gasto_Educacion_PIB', 'Indice_Gini', 'ESCS_media_pais_anual',
       'media_math_estrato', 'media_read_estrato', 'media_scie_estrato',
       'tiempo_deberes_totales_diarios', 'dispone_personal_apoyo'],
      dtype='str')